In [1]:
# ============================================================
# DAY 6 – END-TO-END ML PIPELINE & BUSINESS INSIGHTS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE_PATH = Path("..")

DATA_PATH = BASE_PATH / "data"
PROCESSED_PATH = DATA_PATH / "processed"


# ------------------------------------------------------------
# 2. LOAD PROCESSED DATASETS
# ------------------------------------------------------------

print("=" * 70)
print("LOADING PROCESSED DATA")
print("=" * 70)

files_to_load = {
    "customer_features": "customer_features.csv",
    "customer_segments": "customer_segments.csv",
    "customer_cluster_profile": "customer_cluster_profile.csv",
    "customer_dbscan_profile": "customer_dbscan_profile.csv",
    "restaurant_features": "restaurant_features.csv",
    "reviews_sentiment": "reviews_sentiment.csv",
    "hourly_demand": "hourly_demand.csv",
    "forecast_comparison": "demand_forecast_model_comparison.csv",
    "final_forecast": "final_demand_forecast.csv"
}

datasets = {}

for name, filename in files_to_load.items():

    filepath = PROCESSED_PATH / filename

    if filepath.exists():
        datasets[name] = pd.read_csv(filepath)
        print(f"{name:<30} {datasets[name].shape}")
    else:
        print(f"{name:<30} FILE NOT FOUND: {filepath}")


# ------------------------------------------------------------
# 3. CUSTOMER SEGMENTATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CUSTOMER SEGMENTATION")
print("=" * 70)

if "customer_segments" in datasets:

    customer_segments = datasets["customer_segments"]

    print("\nCustomer segment columns:")
    print(customer_segments.columns.tolist())

    print("\nCustomer segment distribution:")

    # Automatically identify likely cluster column
    cluster_columns = [
        col for col in customer_segments.columns
        if "cluster" in col.lower()
    ]

    if cluster_columns:

        cluster_col = cluster_columns[0]

        print(
            customer_segments[cluster_col]
            .value_counts()
            .sort_index()
        )

        customer_segment_distribution = (
            customer_segments[cluster_col]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
        )

        print("\nCustomer segment percentages:")
        print(customer_segment_distribution)


# ------------------------------------------------------------
# 4. RESTAURANT ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESTAURANT ANALYSIS")
print("=" * 70)

if "restaurant_features" in datasets:

    restaurant_features = datasets["restaurant_features"]

    print("\nRestaurant feature columns:")
    print(restaurant_features.columns.tolist())

    print("\nRestaurant summary statistics:")

    print(
        restaurant_features.describe()
        .round(2)
    )

    # Top restaurants by revenue
    if "revenue" in restaurant_features.columns:

        top_restaurants = (
            restaurant_features
            .sort_values("revenue", ascending=False)
            .head(10)
        )

        print("\nTop 10 restaurants by revenue:")
        print(
            top_restaurants[
                [
                    col for col in [
                        "restaurant_id",
                        "revenue",
                        "total_orders",
                        "avg_rating",
                        "cancellation_rate",
                        "avg_delivery_time"
                    ]
                    if col in top_restaurants.columns
                ]
            ]
            .round(2)
        )


# ------------------------------------------------------------
# 5. SENTIMENT ANALYSIS SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SENTIMENT ANALYSIS")
print("=" * 70)

if "reviews_sentiment" in datasets:

    reviews_sentiment = datasets["reviews_sentiment"]

    print("\nSentiment columns:")
    print(reviews_sentiment.columns.tolist())

    if "sentiment" in reviews_sentiment.columns:

        sentiment_distribution = (
            reviews_sentiment["sentiment"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        )

        print("\nReview sentiment distribution:")
        print(sentiment_distribution)

    if "review_rating" in reviews_sentiment.columns:

        print("\nAverage review rating:")
        print(
            round(
                reviews_sentiment["review_rating"].mean(),
                2
            )
        )


# ------------------------------------------------------------
# 6. NEGATIVE REVIEW ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NEGATIVE REVIEW ANALYSIS")
print("=" * 70)

if "reviews_sentiment" in datasets:

    reviews_sentiment = datasets["reviews_sentiment"]

    if "review_rating" in reviews_sentiment.columns:

        negative_reviews = reviews_sentiment[
            reviews_sentiment["review_rating"] <= 2
        ]

        print(
            f"\nNegative reviews (rating <= 2): "
            f"{len(negative_reviews)}"
        )

        print(
            f"Percentage of reviews that are negative: "
            f"{len(negative_reviews) / len(reviews_sentiment) * 100:.2f}%"
        )


# ------------------------------------------------------------
# 7. DELIVERY MODEL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DELIVERY TIME PREDICTION")
print("=" * 70)

delivery_model_results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Ridge",
        "Lasso",
        "Random Forest"
    ],
    "MAE": [
        12.9349,
        4.7928,
        4.7925,
        4.3324
    ],
    "RMSE": [
        16.1913,
        5.9970,
        5.9973,
        5.4652
    ],
    "R2": [
        -0.0001,
        0.8628,
        0.8628,
        0.8861
    ]
})

print("\nDelivery model comparison:")
print(delivery_model_results)


# ------------------------------------------------------------
# 8. DELIVERY MODEL SELECTION
# ------------------------------------------------------------

best_delivery_model = delivery_model_results.loc[
    delivery_model_results["MAE"].idxmin()
]

print("\nBest delivery prediction model:")
print(best_delivery_model)


# ------------------------------------------------------------
# 9. DELIVERY FEATURE IMPORTANCE
# ------------------------------------------------------------

delivery_feature_importance = pd.DataFrame({
    "Feature": [
        "delivery_distance_km",
        "traffic_condition_High",
        "preparation_time_min",
        "hour",
        "traffic_condition_Low",
        "traffic_condition_Medium",
        "weather_Clear",
        "weather_Cloudy",
        "order_amount",
        "weather_Stormy",
        "weather_Rainy",
        "restaurant_rating",
        "avg_prep_time",
        "day_of_week",
        "item_count"
    ],
    "Importance": [
        0.264296,
        0.225920,
        0.223347,
        0.053281,
        0.039668,
        0.036166,
        0.032217,
        0.021407,
        0.015458,
        0.015209,
        0.015076,
        0.014695,
        0.012153,
        0.007683,
        0.005332
    ]
})

print("\nTop delivery-time drivers:")
print(
    delivery_feature_importance
    .head(10)
    .round(4)
)


# ------------------------------------------------------------
# 10. DEMAND FORECASTING SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DEMAND FORECASTING")
print("=" * 70)

if "forecast_comparison" in datasets:

    forecast_comparison = datasets["forecast_comparison"]

    print("\nForecast model comparison:")
    print(forecast_comparison)


# ------------------------------------------------------------
# 11. FINAL FORECAST SUMMARY
# ------------------------------------------------------------

if "final_forecast" in datasets:

    final_forecast = datasets["final_forecast"].copy()

    print("\nFinal forecast columns:")
    print(final_forecast.columns.tolist())

    forecast_column = None

    for col in final_forecast.columns:

        if "forecast" in col.lower():
            forecast_column = col
            break

    if forecast_column:

        print("\nFinal forecast summary:")
        print(
            final_forecast[forecast_column]
            .describe()
            .round(2)
        )

        print(
            f"\nTotal forecasted orders: "
            f"{final_forecast[forecast_column].sum():.0f}"
        )

        print(
            f"Average hourly forecast: "
            f"{final_forecast[forecast_column].mean():.2f}"
        )

        print(
            f"Peak hourly forecast: "
            f"{final_forecast[forecast_column].max():.2f}"
        )


# ------------------------------------------------------------
# 12. DEMAND PATTERN SUMMARY
# ------------------------------------------------------------

if "hourly_demand" in datasets:

    hourly_demand = datasets["hourly_demand"].copy()

    print("\n" + "=" * 70)
    print("DEMAND PATTERNS")
    print("=" * 70)

    if "order_count" in hourly_demand.columns:

        hourly_demand["hour_timestamp"] = pd.to_datetime(
            hourly_demand["hour_timestamp"]
        )

        hourly_demand["hour"] = (
            hourly_demand["hour_timestamp"].dt.hour
        )

        hourly_demand["day_of_week"] = (
            hourly_demand["hour_timestamp"].dt.dayofweek
        )

        hourly_demand["is_weekend"] = (
            hourly_demand["day_of_week"] >= 5
        )

        peak_hours = (
            hourly_demand
            .groupby("hour")["order_count"]
            .mean()
            .sort_values(ascending=False)
            .head(5)
        )

        print("\nTop 5 peak hours:")
        print(peak_hours.round(2))

        weekend_comparison = (
            hourly_demand
            .groupby("is_weekend")["order_count"]
            .mean()
        )

        print("\nWeekday vs Weekend:")
        print(weekend_comparison.round(2))


# ------------------------------------------------------------
# 13. CONSOLIDATED MODEL COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONSOLIDATED ML MODEL SUMMARY")
print("=" * 70)

model_summary = pd.DataFrame({
    "Component": [
        "Customer Segmentation",
        "Restaurant Segmentation",
        "Sentiment Analysis",
        "Delivery Time Prediction",
        "Demand Forecasting"
    ],
    "Primary_Method": [
        "K-Means",
        "Feature-based segmentation",
        "TF-IDF + Naive Bayes",
        "Random Forest",
        "Prophet"
    ],
    "Primary_Output": [
        "3 customer segments",
        "Restaurant operational profiles",
        "Customer sentiment classification",
        "Delivery time prediction",
        "7-day hourly demand forecast"
    ]
})

print(model_summary.to_string(index=False))


# ------------------------------------------------------------
# 14. SAVE DAY 6 BUSINESS SUMMARY
# ------------------------------------------------------------

model_summary.to_csv(
    PROCESSED_PATH / "day6_model_summary.csv",
    index=False
)

delivery_model_results.to_csv(
    PROCESSED_PATH / "day6_delivery_model_comparison.csv",
    index=False
)

delivery_feature_importance.to_csv(
    PROCESSED_PATH / "day6_delivery_feature_importance.csv",
    index=False
)

print("\n" + "=" * 70)
print("DAY 6 INITIAL INTEGRATION COMPLETE")
print("=" * 70)

LOADING PROCESSED DATA
customer_features              (10000, 8)
customer_segments              (10000, 11)
customer_cluster_profile       (3, 9)
customer_dbscan_profile        (2, 8)
restaurant_features            (500, 7)
reviews_sentiment              (54001, 9)
hourly_demand                  (13081, 6)
forecast_comparison            (2, 4)
final_forecast                 (168, 2)

CUSTOMER SEGMENTATION

Customer segment columns:
['customer_id', 'total_orders', 'avg_order_value', 'total_spending', 'ordering_frequency', 'avg_rating_given', 'weekend_orders', 'late_night_orders', 'cluster', 'customer_segment', 'dbscan_cluster']

Customer segment distribution:
cluster
0    3421
1    2925
2    3654
Name: count, dtype: int64

Customer segment percentages:
cluster
0    34.21
1    29.25
2    36.54
Name: proportion, dtype: float64

RESTAURANT ANALYSIS

Restaurant feature columns:
['restaurant_id', 'total_orders', 'revenue', 'avg_rating', 'avg_preparation_time', 'cancellation_rate', 'avg_deliv